In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *


In [0]:
data = [
(1, 1, 0),
(1, 1, 1),
(1, 1, 2),
(1, 2, 3),
(1, 2, 4),
(2, 1, 5),
(2, 1, 6)
]
schema = StructType([
StructField("ActorId",IntegerType(),True),
StructField("DirectorId",IntegerType(),True),
StructField("timestamp",IntegerType(),True)
])

In [0]:
df = spark.createDataFrame(data, schema)

display(df)

In [0]:
df_group = df.groupBy("ActorId").count()

In [0]:
display(df_group)

In [0]:
df_filter = df_group.filter(df_group['count']>=3)

In [0]:
display(df_filter)

In [0]:
data = [
(1, 1, 'Clicked'),
(2, 2, 'Clicked'),
(3, 3, 'Viewed'),
(5, 5, 'Ignored'),
(1, 7, 'Ignored'),
(2, 7, 'Viewed'),
(3, 5, 'Clicked'),
(1, 4, 'Viewed'),
(2, 11, 'Viewed'),
(1, 2, 'Clicked')
]
schema=StructType([
StructField('AD_ID',IntegerType(),True)
,StructField('USER_ID',IntegerType(),True)
,StructField('ACTION',StringType(),True)
])

In [0]:
df = spark.createDataFrame(data, schema)
display(df)

In [0]:
from pyspark.sql.functions import col, sum as _sum, count, when, round, try_divide, coalesce, lit

ctr_df = (
    df
    .groupBy("AD_ID")
    .agg(
        _sum(when(col("ACTION") == "Clicked", 1).otherwise(0)).alias("clicks"),
        count(when(col("ACTION") != "Ignored", 1)).alias("impressions")
    )
    .withColumn(
        "ctr",
        round(try_divide(col("clicks"), col("impressions")), 2)
    )
    .withColumn(
        "ctr",
        coalesce(col("ctr"), lit(0.0))
    )
    .select("AD_ID", "ctr")
    .orderBy(col("ctr").desc())
)
display(ctr_df)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
